# Calculate unit-wise difference between the Taunton-Worcester scenario and base

Aggregate differences by stroke unit.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.calculations import calculate_extra_bits

In [2]:
dir_output = 'output'
# Scenario results:
file_comparison_unit_combo = 'comparison_by_unit_combo_base_taunworc'
# Output:
file_comparison_nearest_unit = 'comparison_by_nearest_unit_base_taunworc'
file_comparison_transfer_unit = 'comparison_by_transfer_unit_base_taunworc'

In [3]:
strings_tl = ['', '_transferloss0!67']  # column names for transfer rates
scenarios = ['base', 'taunworc']

## Set up column names

Gather columns present in the main results dataframes.

Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [4]:
col_admissions = 'Admissions_2025_Est'
cols_admissions_weighted_outcomes = [
    'admissions_weighted_drip_ship_mrs_0-2',
    'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
]
cols_time = ['drip_ship_mt_time',
             'drip_ship_mt_travel_time',
             'drip_ship_mt_travel_time_with_delay']


# Add columns affected by lossy transfer rate:
def add_str_to_columns(cols, s_list):
    return [f'{c}{s}' for c in cols for s in s_list]

cols_admissions_weighted_outcomes = add_str_to_columns(
    cols_admissions_weighted_outcomes, strings_tl)
cols_time = add_str_to_columns(
    cols_time, strings_tl)

## Function to combine units

Admissions can just be summed.

Admission-weighted outcomes and travel times need to be weighted more carefully.

In [5]:
def combine_units_admissions_weighted_outcomes(
        df,  # by unit, units may appear more than once
        cols_admissions_weighted_outcomes,
        col_admissions,
        col_unit
    ):
    df = df.copy()
    df[cols_admissions_weighted_outcomes] = (
        df[cols_admissions_weighted_outcomes]
            .multiply(df[col_admissions], axis='rows')
    )
    cols = [col_unit, col_admissions] + cols_admissions_weighted_outcomes
    df_here = df[cols].groupby(col_unit).sum()
    # Divide outcomes by the direct admissions to each unit:
    df_here[cols_admissions_weighted_outcomes] = (
        df_here[cols_admissions_weighted_outcomes]
            .div(df_here[col_admissions], axis='rows')
    )
    df_here = df_here.drop(col_admissions, axis='columns')
    df_here = df_here.round(5)

    return df_here

In [6]:
def combine_units_average_times(
        df,  # by unit, units may appear more than once
        cols_time,
        col_admissions,
        col_unit
    ):
    df = df.copy()
    # Convert average time per patient to total time for all patients:
    df[cols_time] = df[cols_time].multiply(df[col_admissions], axis='rows')

    cols = [col_unit, col_admissions] + cols_time
    df_here = df[cols].groupby(col_unit).sum()

    # Divide total times by the total number of patients to each unit:
    df_here[cols_time] = df_here[cols_time].div(
        df_here[col_admissions], axis='rows')

    df_here = df_here.drop(col_admissions, axis='columns')
    df_here = df_here.round(1)

    return df_here

## Run calculations

This cell prints the columns that are created by each function in calculate_results_per_scenario().

In [7]:
for s_bias in ['_bias10min', '']:
    # Load in the LSOA-level outcomes and admissions data:
    p = os.path.join(dir_output, file_comparison_unit_combo + s_bias + '.csv')
    comparison_by_unit_combo = pd.read_csv(p, index_col=0)

    cols_unit = ['nearest_ivt_unit', 'transfer_unit']
    cols_to_sum = []
    
    dict_by_unit = {}
    for s in ['base', 'taunworc']:
        # Set up columns:
        cols_unit_here = [f'{c}_{s}' for c in cols_unit]
        cols_scen = [c for c in comparison_by_unit_combo.columns if s in c]
        cols_adweight = [c for c in cols_scen if 'weighted' in c]
        cols_time_here = [c for c in cols_scen if 'drip_ship_mt_' in c]
        cols_ad = [c for c in cols_scen if
                       ((('dmissions' in c) | ('transfers_mt' in c)) &
                        (c not in cols_adweight))]
        col_ad = f'{col_admissions}_{s}'
        # Remaining columns should be nearest_ivt_unit and transfer_unit
        # and their SSNAP names and hospital cities.
        cols_left = list(set(cols_scen) -
                         (set(cols_adweight) | set(cols_to_sum)))
    
        for col_unit in cols_unit_here:
            # Stroke unit info:
            if 'transfer' in col_unit:
                cols_this_unit = [f'SSNAP name_transfer_unit_{s}',
                                  f'Hospital_City_transfer_unit_{s}']
            else:
                cols_this_unit = [f'SSNAP name_{s}', f'Hospital_City_{s}']
            df_units = (comparison_by_unit_combo[[col_unit] + cols_this_unit]
                .drop_duplicates(ignore_index=True))
            df_units = df_units.set_index(col_unit)
    
            df_ad = comparison_by_unit_combo[[col_unit] + cols_ad].copy()
            df_ad = df_ad.groupby(col_unit).sum()
    
            df_adweight = comparison_by_unit_combo[
                [col_unit, col_ad] + cols_adweight].copy()
            df_adweight = combine_units_admissions_weighted_outcomes(
                df_adweight,  # by unit, units may appear more than once
                cols_adweight,
                col_ad,
                col_unit
            )

            df_times = comparison_by_unit_combo[
                [col_unit, col_ad] + cols_time_here].copy()
            df_times = combine_units_average_times(
                df_times,  # by unit, units may appear more than once
                cols_time_here,
                col_ad,
                col_unit
            )

            df_here = pd.concat((df_units, df_ad, df_adweight, df_times),
                                axis='columns')
    
            dict_by_unit[col_unit] = df_here.copy()
    
    # Combine scenarios.    
    # Keep the nearest IVT unit and transfer unit data separate
    # because they're not always the same pair between scenarios.
    comparison_by_nearest_ivt_unit = pd.concat(
        (dict_by_unit['nearest_ivt_unit_base'],
         dict_by_unit['nearest_ivt_unit_taunworc']),
        axis='columns'
    )
    comparison_by_transfer_unit = pd.concat(
        (dict_by_unit['transfer_unit_base'],
         dict_by_unit['transfer_unit_taunworc']),
        axis='columns'
    )
    
    comparison_by_nearest_ivt_unit = calculate_extra_bits(
        comparison_by_nearest_ivt_unit, scenarios)
    comparison_by_transfer_unit = calculate_extra_bits(
        comparison_by_transfer_unit, scenarios)
    
    p = os.path.join(dir_output, file_comparison_nearest_unit + s_bias + '.csv')
    comparison_by_nearest_ivt_unit.to_csv(p)
    
    p = os.path.join(dir_output, file_comparison_transfer_unit + s_bias + '.csv')
    comparison_by_transfer_unit.to_csv(p)

In [8]:
comparison_by_nearest_ivt_unit

,SSNAP name_base,Hospital_City_base,Admissions_2025_Est_base,admissions_mt_base,admissions_mt_transferloss0!67_base,transfers_mt_base,transfers_mt_transferloss0!67_base,admissions_weighted_drip_ship_mrs_0-2_base,admissions_weighted_drip_ship_mrs_0-2_transferloss0!67_base,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base,...,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_taunworc,drip_ship_mt_time_taunworc,drip_ship_mt_travel_time_taunworc,drip_ship_mt_travel_time_with_delay_taunworc,change_in_first_admissions,change_in_admissions_mt,change_in_admissions_mt_transferloss0!67,change_in_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67,change_in_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2
BN25BE,Royal Sussex County Hospital,Brighton,654.1,65.4,65.4,0.0,0.0,0.61838,0.61838,0.42841,...,0.42841,0.42841,182.5,12.5,12.5,0.0,0.0,0.0,0.0,0.0
BN112DH,Worthing Hospital,Worthing,584.5,58.4,39.1,58.4,39.1,0.60376,0.59866,0.37688,...,0.37688,0.35985,263.9,33.9,93.9,0.0,0.0,0.0,0.0,0.0
CA27HY,Cumberland Infirmary,Carlisle,682.8,68.3,45.7,68.3,45.7,0.58884,0.58666,0.34296,...,0.34296,0.33565,319.9,89.9,149.9,0.0,0.0,0.0,0.0,0.0
NG174JL,Kings Mill Hospital,Mansfield,750.4,75.0,50.2,75.0,50.2,0.60433,0.59909,0.37834,...,0.37834,0.36084,261.6,31.6,91.6,0.0,0.0,0.0,0.0,0.0
DE223NE,Royal Derby Hospital,Derby,1052.7,105.2,70.5,105.2,70.5,0.60337,0.59814,0.37757,...,0.37757,0.36012,262.8,32.8,92.8,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PO305TG,St Mary's Hospital Newport,Newport,304.8,30.5,20.4,30.5,20.4,0.58779,0.58752,0.32699,...,0.32699,0.32607,347.0,117.0,177.0,0.0,0.0,0.0,0.0,0.0
L97AL,University Hospital Aintree,Aintree,1573.5,157.3,157.3,0.0,0.0,0.61836,0.61836,0.42842,...,0.42842,0.42842,182.5,12.5,12.5,0.0,0.0,0.0,0.0,0.0
NE236NZ,Northumbria Specialist Emergency Care Hospital...,Cramlington,781.5,78.1,52.3,78.1,52.3,0.60296,0.59745,0.37951,...,0.37951,0.36113,259.8,29.8,89.8,0.0,0.0,0.0,0.0,0.0
TF16TF,Princess Royal Hospital Telford,Telford,927.8,92.8,62.1,92.8,62.1,0.59835,0.59427,0.36487,...,0.36487,0.35124,283.5,53.5,113.5,0.0,0.0,0.0,0.0,0.0
